# Proyecto BI – Purchasing | Sprint 1

**Proyecto:** Análisis de compras y proveedores de AdventureWorks  
**Sprint:** 1 – Extracción, preparación y análisis exploratorio de datos (EDA)  
**Integrantes:** [Miguel Ángel Ortega Hernandez, José Manuel Molina, Sebastian Agudelo]  
**Fecha:** [14/10/2026]


## 1. Objetivo del sprint

Obtener y preparar los datos relacionados con compras y proveedores de AdventureWorks para comprender su estructura y calidad, realizar el análisis exploratorio de datos (EDA) y dejar una base de trabajo para las siguientes etapas del proyecto, incluida la posible segmentación de proveedores mediante K-Means.

En este notebook se documentan la fuente de los datos, las tablas utilizadas y el proceso de preparación. Los hallazgos del EDA y las decisiones sobre las variables de segmentación se incorporarán cuando el equipo complete su análisis.

## 2. Fuente y alcance de los datos

Los datos se obtuvieron de la base de datos AdventureWorks mediante una conexión de Python a SQL Server. Las tablas se exportaron a archivos CSV para facilitar su revisión y análisis local.

### Tablas extraídas

| Tabla | Descripción general |
|---|---|
| `Purchasing.Vendor` | Información principal de los proveedores. |
| `Purchasing.PurchaseOrderHeader` | Información general de las órdenes de compra, proveedor, fechas y valores. |
| `Purchasing.PurchaseOrderDetail` | Líneas de detalle de las órdenes, productos, cantidades y precios. |
| `Purchasing.ProductVendor` | Relación entre proveedores y productos, con información como precio y tiempo promedio de entrega. |
| `Purchasing.ShipMethod` | Catálogo de métodos de envío. |
| `Production.Product` | Información descriptiva de los productos. |

### Archivos generados

- `data/raw/vendor.csv`
- `data/raw/purchase_order_header.csv`
- `data/raw/purchase_order_detail.csv`
- `data/raw/product_vendor.csv`
- `data/raw/ship_method.csv`
- `data/raw/product.csv`
- `data/processed/purchasing_consolidated.csv`

El archivo consolidado reúne indicadores a nivel de proveedor. Los CSV originales se conservan por separado para mantener una copia de los datos extraídos.

## 3. Estructura del repositorio

```text
bi-purchasing/
├── data/
│   ├── raw/          # CSV originales extraídos de SQL Server
│   └── processed/    # Dataset consolidado
├── notebooks/
│   └── 01_purchasing_eda.ipynb
├── src/
│   ├── db.py
│   ├── extract_data.py
│   ├── inspect_data.py
│   └── consolidate_data.py
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md
```

La estructura puede actualizarse si el equipo incorpora nuevos scripts o archivos.

## 4. Metodología de trabajo

1. **Conexión y extracción:** conexión a SQL Server y exportación de las tablas seleccionadas a CSV.
2. **Inspección inicial:** revisión del tamaño de las tablas, columnas, tipos de datos y valores nulos.
3. **Consolidación:** relación de los datos y generación de indicadores por proveedor, evitando duplicaciones en los valores de las órdenes.
4. **EDA:** análisis de calidad, estadísticas descriptivas, distribuciones y relaciones relevantes entre variables.
5. **Preparación posterior:** selección y preparación de variables para evaluar la segmentación de proveedores.

La selección definitiva de variables y la interpretación de los grupos se definirán a partir de los resultados del análisis y los acuerdos del equipo.

## 5. Preparación realizada

Hasta el momento se completaron las siguientes actividades:

- Se verificó la conexión desde Python a la base de datos.
- Se extrajeron seis tablas a archivos CSV.
- Se revisó inicialmente el tamaño de las tablas, las columnas y los valores nulos.
- Se generó `purchasing_consolidated.csv` como una primera consolidación a nivel de proveedor.

La consolidación es una base de trabajo inicial y debe validarse antes de utilizarse para análisis estadístico o modelado.

## 6. Resumen del análisis exploratorio de datos (EDA)

El análisis detallado de estructura, calidad y estadística fue desarrollado por Miguel en su notebook **EDA Parte 1 – Estructura, calidad y estadística**. En esta sección se integran los hallazgos principales para dar una visión general del sprint; las tablas, gráficos y cálculos completos permanecen en el notebook de EDA.

### 6.1 Estructura general

El análisis consideró las cuatro tablas principales del esquema Purchasing y el dataset consolidado por proveedor:

| Dataset | Filas | Columnas |
|---|---:|---:|
| `Vendor` | 106 | 5 |
| `PurchaseOrderHeader` | 4.019 | 13 |
| `PurchaseOrderDetail` | 8.853 | 11 |
| `ProductVendor` | 467 | 11 |
| Consolidado por proveedor | 106 | 15 |

El consolidado conserva una fila por proveedor. De los 106 proveedores registrados, 88 tienen información de órdenes que permite calcular las métricas de compra analizadas.

### 6.2 Calidad de los datos y anomalías

- No se encontraron valores faltantes ni filas duplicadas en `Vendor`, `PurchaseOrderHeader` ni `PurchaseOrderDetail`.
- En `ProductVendor` hay 7 valores faltantes en `LastReceiptCost`, 7 en `LastReceiptDate` y 311 en `OnOrderQty` (66,6 %). Estos faltantes deben interpretarse según su significado de negocio; no se utilizan las columnas afectadas para la segmentación propuesta.
- **18 proveedores (16,98 %) no tienen órdenes de compra** y tampoco aparecen asociados a productos en `ProductVendor`. Sus métricas de compra quedan vacías en el consolidado, por lo que no se pueden ubicar en segmentos basados en monto y cantidad de órdenes.
- Se identificó una orden con `TotalDue` igual o menor que cero: la orden 6013, pendiente y sin valor. Se documenta como anomalía para que el equipo decida si debe excluirse de los cálculos.
- Las fechas están almacenadas como texto en los archivos analizados. Deben convertirse a tipo fecha si se realizan análisis temporales.

### 6.3 Distribución y valores atípicos

Los montos de compra presentan asimetría hacia la derecha: la mayoría de los proveedores registra importes menores y unos pocos concentran montos elevados. En `ValorTotalOrdenes`, la media reportada es aproximadamente 801 mil y la mediana aproximadamente 102 mil; el máximo ronda los 5,03 millones.

El análisis mediante IQR identificó valores atípicos en varias variables. Los montos elevados no se consideran automáticamente errores: pueden representar proveedores con una actividad de compra legítimamente alta. Por eso, el notebook de Miguel recomienda conservarlos y revisar su efecto en la segmentación. Algunas variables discretas o casi constantes también aparecen como atípicas por tener un IQR igual a cero, por lo que esos casos no deben eliminarse automáticamente.

### 6.4 Relaciones entre variables y consecuencias para el clustering

La matriz de correlación se calculó sobre los 88 proveedores con métricas de compra disponibles. Los principales hallazgos fueron:

- `ValorTotalOrdenes` y `ValorLineas` tienen correlación de 1,00, por lo que aportan información prácticamente redundante.
- `ProductosComprados` y `ProductosOfrecidos` también tienen correlación de 1,00 en el análisis.
- `NumeroOrdenes` y `ValorTotalOrdenes` presentan correlación baja (r = 0,17), por lo que describen aspectos distintos del comportamiento de compra.
- La cantidad de órdenes varía poco para la mayoría de los proveedores, mientras que los montos muestran una dispersión considerable.

Con base en estos hallazgos, se recomienda evaluar `ValorTotalOrdenes` y `NumeroOrdenes` como variables principales del clustering, de acuerdo con el alcance definido para el sprint. La decisión definitiva corresponde a la etapa de modelado de Molina.

### 6.5 Decisiones y limitaciones

- Mantener identificados los 18 proveedores sin órdenes, pero excluirlos del conjunto que se use para entrenar K-Means, ya que no cuentan con monto ni número de órdenes.
- Conservar los valores altos de compra mientras no exista evidencia de que sean errores.
- Revisar cómo tratar la orden pendiente con `TotalDue` igual a cero antes de cerrar la consolidación.
- Tener en cuenta que incluir órdenes pendientes o rechazadas en los montos podría afectar las métricas. Miguel señala esta limitación para que el equipo la revise antes de interpretar los resultados.

Para el detalle de los cálculos, las tablas de calidad, los gráficos y las estadísticas, consultar el notebook de Miguel.


## 7. Preparación y resultados de la segmentación de proveedores

La segmentación se realizó a partir del análisis exploratorio y la preparación de variables documentados en los notebooks de Molina. El objetivo fue agrupar proveedores según dos dimensiones de su comportamiento de compra: el monto total comprado y la cantidad de órdenes realizadas.

### 7.1 Preparación de los datos

Para el modelado se utilizaron **88 proveedores con métricas de compra disponibles**. Se seleccionaron las variables `ValorTotalOrdenes` y `NumeroOrdenes`, porque representan dimensiones distintas del comportamiento de compra. La correlación entre ellas es baja (Pearson = 0,168).

Como los montos presentan una fuerte asimetría hacia la derecha, se aplicó `log1p` a `ValorTotalOrdenes`. Después, se utilizó `StandardScaler` para escalar las variables antes de aplicar K-Means, evitando que las diferencias de escala dominaran el cálculo de distancias.

El dataset preparado se guardó localmente como `data/processed/purchasing_clustering_final.csv`, junto con el escalador `scaler_clustering.joblib`. La carpeta `data/processed/` está excluida de Git; cada integrante debe generar los archivos siguiendo las instrucciones de los notebooks de preparación.

### 7.2 Selección del número de clusters

Se evaluaron diferentes valores de `k` mediante el método del codo y el coeficiente de silhouette.

El método del codo identificó **k = 3** como punto de referencia. El silhouette promedio fue de **0,720 para k = 3** y **0,738 para k = 4**. Aunque cuatro clusters obtuvo una puntuación ligeramente superior, produjo un grupo de solo dos proveedores (2,3 % de la población analizada).

Por ello, se seleccionó **k = 3**, considerando conjuntamente el método del codo, la pequeña diferencia en silhouette, el tamaño de los grupos y su interpretabilidad.

### 7.3 Segmentos obtenidos

K-Means produjo los siguientes tres segmentos para los 88 proveedores incluidos en el modelo:

| Segmento | Proveedores | Porcentaje de proveedores | Porcentaje del monto total |
|---|---:|---:|---:|
| Grandes recurrentes | 40 | 45,5 % | 95,0 % |
| Pequeños recurrentes | 39 | 44,3 % | 1,6 % |
| Ocasionales | 9 | 10,2 % | 3,4 % |
| **Total** | **88** | **100 %** | **100 %** |

**Grandes recurrentes.** Agrupa proveedores con muchas órdenes (aproximadamente 51 por proveedor) y montos elevados. El monto mediano es de aproximadamente 1,59 millones y este grupo concentra el 95 % del monto total analizado.

**Pequeños recurrentes.** Incluye proveedores con una frecuencia de órdenes similar (aproximadamente 51), pero con montos mucho menores. El monto mediano es de aproximadamente 25.733 y representan el 1,6 % del monto total.

**Ocasionales.** Reúne nueve proveedores con entre una y cuatro órdenes. Sus montos son heterogéneos —desde valores muy bajos hasta aproximadamente 1,19 millones—, por lo que el rasgo que más los distingue es la baja frecuencia de compra, no un tamaño económico uniforme.

### 7.4 Evaluación de las categorías Micro, Pequeñas, Medianas y Grandes

La idea inicial era clasificar a los proveedores en cuatro categorías ordenadas por tamaño. Sin embargo, los resultados **no justifican interpretar los clusters como Micro, Pequeñas, Medianas y Grandes**.

La solución con cuatro clusters crea un grupo de solo dos proveedores. Además, al comparar la segmentación con categorías definidas por cuartiles del monto, el Adjusted Rand Index (ARI) fue de 0,418, lo que indica que los grupos de K-Means no reproducen esa clasificación de tamaño. El modelo considera tanto el monto como la frecuencia de órdenes.

Por tanto, se conservaron tres etiquetas descriptivas: **Grandes recurrentes, Pequeños recurrentes y Ocasionales**. No representan una clasificación legal ni formal del tamaño empresarial.

### 7.5 Limitaciones de la segmentación

- El modelo utiliza solo el monto total comprado y la cantidad de órdenes; no incluye precio unitario, calidad, tiempos de entrega ni otras características del proveedor.
- La cantidad de órdenes está muy concentrada: el 88,6 % de los proveedores tiene entre 50 y 54 órdenes, por lo que el monto influye considerablemente en la separación.
- Los montos incluyen órdenes pendientes y rechazadas, porque el consolidado no filtra por `Status`. Esta decisión puede afectar los resultados y debe revisarse.
- Los proveedores sin métricas de compra quedaron fuera del modelo y no recibieron un segmento.
- Los nombres de los segmentos son descriptivos y no corresponden a umbrales empresariales validados.

El detalle de la selección y transformación de variables, las métricas de evaluación, los gráficos y la tabla de proveedores clasificados se encuentra en los notebooks de Molina. La tabla final se guarda como `data/processed/purchasing_segmentos_kmeans.csv`.


## 8. Conclusiones y próximos pasos

Durante el Sprint 1 se extrajeron y consolidaron los datos de compras y proveedores de AdventureWorks2025, se revisó su calidad mediante el análisis exploratorio y se prepararon variables para la segmentación.

El análisis con K-Means identificó tres grupos de proveedores con patrones diferentes: **Grandes recurrentes**, **Pequeños recurrentes** y **Ocasionales**. Los primeros concentran el 95 % del monto total analizado; los segundos presentan una frecuencia de órdenes similar, pero montos mucho menores; y los ocasionales se distinguen principalmente por realizar pocas órdenes.

Se eligieron tres clusters porque el método del codo apuntó a `k = 3` y la mejora del silhouette al pasar a cuatro clusters fue pequeña, mientras que la solución de cuatro grupos aislaba únicamente a dos proveedores. Por esta razón, los resultados no respaldan la interpretación de cuatro categorías de tamaño empresarial —Micro, Pequeñas, Medianas y Grandes—. Los segmentos obtenidos describen el comportamiento de compra según monto y frecuencia, no una clasificación legal del tamaño de cada proveedor.

Como próximos pasos, el equipo debe revisar el efecto de las órdenes pendientes y rechazadas sobre los montos, confirmar el tratamiento de las anomalías identificadas durante el EDA y comprobar que los notebooks y archivos de salida puedan reproducirse. Finalmente, Sebastian integrará los hallazgos, verificará que las conclusiones estén respaldadas por los resultados y preparará la presentación del Sprint 1.

En conjunto, el trabajo proporciona una primera segmentación exploratoria que permite identificar patrones de gasto y frecuencia entre los proveedores analizados, con las limitaciones documentadas en el notebook de Molina.
